In [29]:
#!/user/bin/env python3

! pip install biopython
! pip install -q condacolab
import condacolab
condacolab.install()
! conda install -c bioconda seqkit


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.9/16.9 MB 92.9 MB/s eta 0:00:00


✨🍰✨ Everything looks OK!
Channels:
 - bioconda
 - conda-forge
Platform: linux-64
Solving environment: \ | / done


==> WARNING: A newer version of conda exists. <==
    current version: 24.11.3
    latest version: 26.1.1

Please update conda by running

    $ conda update -n base -c conda-forge conda



# All requested packages already installed.



In [27]:
import requests
import shutil
import subprocess
import sys
import builtins
import re
from Bio import SeqIO

class MyFastaParser:
    def __init__(self, file_name):
        self.filename = file_name

    def _get_uniprot(self, accession):
        headers = {"accept": "application/json"}
        url = f"https://rest.uniprot.org/uniprotkb/{accession}"
        return requests.get(url, headers=headers)

    def _get_ensembl(self, id):
        headers = {"accept": "application/json"}
        url = f"https://rest.ensembl.org/lookup/id/{id}"
        return requests.get(url, headers=headers)

    @staticmethod
    def _uniprot_parse_response(resp):
        if not resp.ok:
            resp.raise_for_status()
        data = resp.json()
        return {
            "species": data.get("organism", {}).get("scientificName", ""),
            "geneInfo": data.get("genes", []),
            "sequence": data.get("sequence", {}),
            "type": "protein",
        }

    @staticmethod
    def _ensembl_parse_response(resp):
        if not resp.ok:
            try:
                error_detail = resp.json()
            except:
                error_detail = resp.text
            raise Exception(f"Ensembl API error: {resp.status_code} - {error_detail}")
        data = resp.json()
        attrs = ["object_type", "assembly_name", "species", "db_type",
                 "biotype", "display_name", "description",
                 "canonical_transcript", "source"]
        return {a: data[a] for a in attrs if a in data}

    def _access_database(self, id, database, seq_description, seq_sequence):
        output = {"seq_description": seq_description, "seq_sequence": seq_sequence}
        if database is None:
            output["error"] = "Unknown database format, API lookup skipped"
            return output

        try:
            if database.lower() == "uniprot":
                resp = self._get_uniprot(id)
                if resp.ok:
                    output.update(self._uniprot_parse_response(resp))
                else:
                    output["error"] = f"UniProt request failed: {resp.status_code}"
            elif database.lower() == "ensembl":
                resp = self._get_ensembl(id)
                if resp.ok:
                    output.update(self._ensembl_parse_response(resp))
                else:
                    output["error"] = f"Ensembl request failed: {resp.status_code}"
        except Exception as e:
            output["error"] = f"Exception during request: {e}"
        return output

    def seqkit_stats(self):
        try:
            proc = subprocess.run(
                ("seqkit", "stats", self.filename, "-a"),
                capture_output=True, text=True, check=False
            )
            if proc.stderr:
                return {"ERROR": proc.stderr}
            lines = proc.stdout.strip().split('\n')
            if len(lines) < 2:
                return {"ERROR": "bad output from seqkit"}
            names = lines[0].split()[1:]
            values = lines[1].split()[1:]
            return dict(zip(names, values))
        except Exception as e:
            return {"ERROR": f"exception running seqkit: {e}"}

    def _biopython_stats(self):
        try:
            records = list(SeqIO.parse(self.filename, "fasta"))
            if not records:
                return {"ERROR": "No sequences found"}
            lengths = [len(rec) for rec in records]
            return {
                "num_seqs": builtins.str(len(records)),
                "min_len": builtins.str(min(lengths)),
                "max_len": builtins.str(max(lengths)),
                "avg_len": f"{sum(lengths)/len(lengths):.2f}",
                "total_len": builtins.str(sum(lengths)),
                "format": "fasta"
            }
        except Exception as e:
            return {"ERROR": f"Biopython stats failed: {e}"}


    def biopython_parser(self, seqkit_result):
      if "ERROR" in seqkit_result:
          return seqkit_result

      ext = seqkit_result.get('format', 'fasta').lower()
      output = {}
      try:
          sequences = SeqIO.parse(self.filename, ext)
      except Exception as e:
          return {"ERROR": f"Failed to parse file: {e}"}

      for seq in sequences:
          header = seq.description
          seq_str = builtins.str(seq.seq)
          if header.upper().startswith('ENS'):
              database = 'ensembl'
              identifier = header.split()[0]
              if '.' in identifier:
                  identifier = identifier.split('.')[0]
          elif header.startswith('sp|') or header.startswith('tr|'):
              database = 'uniprot'
              parts = header.split('|')
              identifier = parts[1] if len(parts) >= 2 else header.split()[0]
          else:
              pattern = r'[OPQ][0-9][A-Z0-9]{3}[0-9]|[A-NR-Z][0-9]([A-Z][A-Z0-9]{2}[0-9]){1,2}'
              match = re.search(pattern, header)
              if match:
                  database = 'uniprot'
                  identifier = match.group(0)
              else:
                  database = None
                  identifier = None

          info = self._access_database(identifier, database, header, seq_str)
          output[seq.id] = {
              "header": header,
              "sequence": seq_str,
              "database_info": info
          }
      return output

    def show_output(self, output, indent=0):

      for seq_id, record in output.items():
        header = record["header"]
        sequence = record["sequence"]
        db_info = record["database_info"]

        if header.startswith(('sp|', 'tr|')):
            db_name = "uniprot"
            parts = header.split('|')
            accession = parts[1] if len(parts) >= 2 else seq_id
        elif header.upper().startswith('ENS'):
            db_name = "ensembl"
            accession = header.split()[0].split('.')[0]
        else:
            import re
            pattern = r'[OPQ][0-9][A-Z0-9]{3}[0-9]|[A-NR-Z][0-9]([A-Z][A-Z0-9]{2}[0-9]){1,2}'
            match = re.search(pattern, header)
            if match:
                db_name = "uniprot"
                accession = match.group(0)
            else:
                db_name = "unknown"
                accession = seq_id

        print(f"DB_name\n  {db_name}")

        print(f"file_info_{accession}")
        print(f"  description\n    {header}")
        print(f"  sequence\n    {sequence}")

        print(f"database_info_{accession}")
        if "error" in db_info:
            print(f"  error\n    {db_info['error']}")
        else:
            if "species" in db_info:
                print(f"  organism\n    {db_info['species']}")
            if "geneInfo" in db_info:
                print(f"  geneInfo\n    {db_info['geneInfo']}")
            if "sequence" in db_info and isinstance(db_info["sequence"], dict):
                print(f"  sequenceInfo")
                for subkey, subvalue in db_info["sequence"].items():
                    print(f"    {subkey}\n      {subvalue}")
            if "type" in db_info:
                print(f"  type\n    {db_info['type']}")

        if "error" in db_info:
            print("WARNING")
            print(f"  {db_info['error']}")

        print()


In [28]:
parser = MyFastaParser('test_file.fasta')
stats = parser.seqkit_stats()
print(stats)

biopython = parser.biopython_parser(stats)
parser.show_output(biopython)

{'format': 'FASTA', 'type': 'Protein', 'num_seqs': '2', 'sum_len': '456', 'min_len': '29', 'avg_len': '228', 'max_len': '427', 'Q1': '29', 'Q2': '228', 'Q3': '427', 'sum_gap': '0', 'N50': '427', 'N50_num': '1', 'Q20(%)': '0', 'Q30(%)': '0', 'AvgQual': '0', 'GC(%)': '0', 'sum_n': '0'}
DB_name
  uniprot
file_info_P11473
  description
    sp|P11473|VDR_HUMAN Vitamin D3 receptor OS=Homo sapiens OX=9606 GN=VDR PE=1 SV=1
  sequence
    MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFHFNAMTCEGCKGFFRRSMKRKALFTCPFNGDCRITKDNRRHCQACRLKRCVDIGMMKEFILTDEEVQRKREMILKRKEEEALKDSLRPKLSEEQQRIIAILLDAHHKTYDPTYSDFCQFRPPVRVNDGGGSHPSRPNSRHTPSFSGDSSSSCSDHCITSSDMMDSSSFSNLDLSEEDSDDPSVTLELSQLSMLPHLADLVSYSIQKVIGFAKMIPGFRDLTSEDQIVLLKSSAIEVIMLRSNESFTMDDMSWTCGNQDYKYRVSDVTKAGHSLELIEPLIKFQVGLKKLNLHEEEHVLLMAICIVSPDRPGVQDAALIEAIQDRLSNTLQTYIRCRHPPPGSHLLYAKMIQKLADLRSLNEEHSKQYRCLSFQPECSMKLTPLVLEVFGNEIS
database_info_P11473
  organism
    Homo sapiens
  geneInfo
    [{'geneName': {'evidences': [{'evidenceCode': 'ECO:0000312', 'source': 'HG

In [29]:
parser1 = MyFastaParser('ensembl_download_1.fasta')
stats1 = parser1.seqkit_stats()
print(stats1)
biopython1 = parser1.biopython_parser(stats1)

parser1.show_output(biopython1)

{'format': 'FASTA', 'type': 'DNA', 'num_seqs': '6', 'sum_len': '86', 'min_len': '9', 'avg_len': '14.3', 'max_len': '23', 'Q1': '10', 'Q2': '13.5', 'Q3': '17', 'sum_gap': '0', 'N50': '16', 'N50_num': '3', 'Q20(%)': '0', 'Q30(%)': '0', 'AvgQual': '0', 'GC(%)': '45.35', 'sum_n': '0'}
DB_name
  ensembl
file_info_ENSMUST00000196221
  description
    ENSMUST00000196221.2 cds chromosome:GRCm39:14:54350925:54350933:1 gene:ENSMUSG00000096749.3 gene_biotype:TR_D_gene transcript_biotype:TR_D_gene gene_symbol:Trdd1 description:T cell receptor delta diversity 1 [Source:MGI Symbol;Acc:MGI:4439547]
  sequence
    ATGGCATAT
database_info_ENSMUST00000196221
  organism
    mus_musculus

DB_name
  ensembl
file_info_ENSMUST00000177564
  description
    ENSMUST00000177564.2 cds chromosome:GRCm39:14:54359683:54359698:1 gene:ENSMUSG00000096176.2 gene_biotype:TR_D_gene transcript_biotype:TR_D_gene gene_symbol:Trdd2 description:T cell receptor delta diversity 2 [Source:MGI Symbol;Acc:MGI:4439546]
  sequence
  

In [30]:
parser2 = MyFastaParser('uniprot_download.fasta')
stats2 = parser2.seqkit_stats()
print(stats2)
biopython2 = parser2.biopython_parser(stats2)

parser2.show_output(biopython)

{'format': 'FASTA', 'type': 'Protein', 'num_seqs': '7', 'sum_len': '3,861', 'min_len': '180', 'avg_len': '551.6', 'max_len': '1,382', 'Q1': '429', 'Q2': '441', 'Q3': '500', 'sum_gap': '0', 'N50': '468', 'N50_num': '3', 'Q20(%)': '0', 'Q30(%)': '0', 'AvgQual': '0', 'GC(%)': '0', 'sum_n': '0'}
DB_name
  uniprot
file_info_P11473
  description
    sp|P11473|VDR_HUMAN Vitamin D3 receptor OS=Homo sapiens OX=9606 GN=VDR PE=1 SV=1
  sequence
    MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFHFNAMTCEGCKGFFRRSMKRKALFTCPFNGDCRITKDNRRHCQACRLKRCVDIGMMKEFILTDEEVQRKREMILKRKEEEALKDSLRPKLSEEQQRIIAILLDAHHKTYDPTYSDFCQFRPPVRVNDGGGSHPSRPNSRHTPSFSGDSSSSCSDHCITSSDMMDSSSFSNLDLSEEDSDDPSVTLELSQLSMLPHLADLVSYSIQKVIGFAKMIPGFRDLTSEDQIVLLKSSAIEVIMLRSNESFTMDDMSWTCGNQDYKYRVSDVTKAGHSLELIEPLIKFQVGLKKLNLHEEEHVLLMAICIVSPDRPGVQDAALIEAIQDRLSNTLQTYIRCRHPPPGSHLLYAKMIQKLADLRSLNEEHSKQYRCLSFQPECSMKLTPLVLEVFGNEIS
database_info_P11473
  organism
    Homo sapiens
  geneInfo
    [{'geneName': {'evidences': [{'evidenceCode': 'ECO:0000312', 'sour

In [31]:
parser3 = MyFastaParser("ensembl_download_2.fasta")
stats3 = parser3.seqkit_stats()
print(stats3)
biopython3 = parser3.biopython_parser(stats3)

parser3.show_output(biopython)

{'ERROR': '\x1b[ERRO]\x1b ensembl_download_2.fasta: fastx: invalid FASTA/Q format\n'}
DB_name
  uniprot
file_info_P11473
  description
    sp|P11473|VDR_HUMAN Vitamin D3 receptor OS=Homo sapiens OX=9606 GN=VDR PE=1 SV=1
  sequence
    MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFHFNAMTCEGCKGFFRRSMKRKALFTCPFNGDCRITKDNRRHCQACRLKRCVDIGMMKEFILTDEEVQRKREMILKRKEEEALKDSLRPKLSEEQQRIIAILLDAHHKTYDPTYSDFCQFRPPVRVNDGGGSHPSRPNSRHTPSFSGDSSSSCSDHCITSSDMMDSSSFSNLDLSEEDSDDPSVTLELSQLSMLPHLADLVSYSIQKVIGFAKMIPGFRDLTSEDQIVLLKSSAIEVIMLRSNESFTMDDMSWTCGNQDYKYRVSDVTKAGHSLELIEPLIKFQVGLKKLNLHEEEHVLLMAICIVSPDRPGVQDAALIEAIQDRLSNTLQTYIRCRHPPPGSHLLYAKMIQKLADLRSLNEEHSKQYRCLSFQPECSMKLTPLVLEVFGNEIS
database_info_P11473
  organism
    Homo sapiens
  geneInfo
    [{'geneName': {'evidences': [{'evidenceCode': 'ECO:0000312', 'source': 'HGNC', 'id': 'HGNC:12679'}], 'value': 'VDR'}, 'synonyms': [{'value': 'NR1I1'}]}]
  sequenceInfo
    value
      MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFHFNAMTCEGCKGFFRRSMKRKALFTCPFNGDCRITKDNRRHCQACRLKRCVDIGM